In [1]:
import os
import glob
import pandas as pd

base_dir = r"Z:\HTOC\Data_Analytics\Data\OpDiv_Predictions\Full Daily Reports"

# Adjust the pattern if filenames have a specific pattern, e.g. "*_daily.csv"
csv_files = glob.glob(os.path.join(base_dir, "*.csv"))

dfs = []
for f in csv_files:
    df = pd.read_csv(f)
    df["source_file"] = os.path.basename(f)  # optional, to track origin
    dfs.append(df)

all_data = pd.concat(dfs, ignore_index=True)
all_data

,Indicator,Observed Today,Frequency (7d),Frequency (30d),Probability: 7-Day,Confidence: 7-Day,Probability: 14-Day,Confidence: 14-Day,Probability: 30-Day,Confidence: 30-Day,Partner,FileDate,source_file,ensemble_45d,Confidence: 45-Day,Frequency (1d),Probability: 1-Day,Confidence: 1-Day
0,102.129.153.158,0,0.0,1.0,6.88%,7-Day: Low confidence,13.68%,14-Day: Low confidence,68.66%,30-Day: Possibly active,CDC,20250616.0,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
1,102.129.153.43,0,0.0,0.0,0.38%,7-Day: Low confidence,0.93%,14-Day: Low confidence,18.11%,30-Day: Low confidence,CDC,20250616.0,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
2,102.129.153.71,0,0.0,2.0,12.39%,7-Day: Low confidence,24.8%,14-Day: Low confidence,86.71%,30-Day: Highly likely,CDC,20250616.0,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
3,102.165.16.161,0,0.0,0.0,0.01%,7-Day: Low confidence,0.01%,14-Day: Low confidence,0.17%,30-Day: Low confidence,CDC,20250616.0,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
4,103.133.107.28,0,0.0,1.0,9.14%,7-Day: Low confidence,59.14%,14-Day: Possibly active,73.35%,30-Day: Possibly active,CDC,20250616.0,full_daily_report_20250616.csv,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2446217,teletype.in,0,0.0,2.0,8.69%,7-Day: Low confidence,16.82%,14-Day: Low confidence,69.88%,30-Day: Highly likely,VA,20260322.0,full_daily_report_20260322.csv,52.97%,45-Day: Possibly active,0.0,1.32%,1-Day: Low confidence
2446218,tester.com,0,0.0,0.0,0.0%,7-Day: Low confidence,0.0%,14-Day: Low confidence,0.13%,30-Day: Low confidence,VA,20260322.0,full_daily_report_20260322.csv,0.58%,45-Day: Low confidence,0.0,0.01%,1-Day: Low confidence
2446219,www.citiquartz.com,0,0.0,0.0,0.38%,7-Day: Low confidence,0.89%,14-Day: Low confidence,15.6%,30-Day: Low confidence,VA,20260322.0,full_daily_report_20260322.csv,29.49%,45-Day: Possibly active,0.0,0.02%,1-Day: Low confidence
2446220,www.deepseek.com,0,0.0,12.0,20.08%,7-Day: Low confidence,27.45%,14-Day: Low confidence,80.46%,30-Day: Highly likely,VA,20260322.0,full_daily_report_20260322.csv,86.08%,45-Day: Highly likely,0.0,6.64%,1-Day: Low confidence


In [2]:
# Clean, convert, and aggregate in one step

# Rename 45-day probability column if present
if "ensemble_45d" in all_data.columns:
    all_data = all_data.rename(columns={"ensemble_45d": "Probability: 45-Day"})

# Drop confidence columns and other unwanted fields
cols_to_drop = [c for c in all_data.columns if "confidence" in c.lower()]
cols_to_drop += ["source_file", "FileDate", "Partner", "Observed Today"]
# Remove duplicates while preserving order
cols_to_drop = list(dict.fromkeys(cols_to_drop))
all_data = all_data.drop(columns=cols_to_drop, errors="ignore")

# Convert probability columns from strings like "12.3%" to numeric 0–1 floats
prob_cols = [c for c in all_data.columns if "Probability" in c]
for c in prob_cols:
    all_data[c] = pd.to_numeric(
        all_data[c].astype(str).str.rstrip("%"), errors="coerce"
    ) / 100.0

# Group by Indicator and take average of numeric columns
grouped = all_data.groupby("Indicator", as_index=False).mean(numeric_only=True)

# Round frequency columns to nearest whole number
freq_cols = [c for c in grouped.columns if c.startswith("Frequency ")]
for c in freq_cols:
    grouped[c] = grouped[c].round(0).astype("Int64")

# Convert probabilities (0–1) to percentages with 2 decimals
prob_cols_grouped = [c for c in grouped.columns if "Probability" in c]
for c in prob_cols_grouped:
    grouped[c] = (grouped[c] * 100).round(2)    

grouped

,Indicator,Frequency (7d),Frequency (30d),Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day,Frequency (1d),Probability: 1-Day
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0,0,9.21,18.59,27.13,29.95,0,0.34
1,1-you.njalla.no,0,2,23.79,39.99,61.14,48.66,0,8.62
2,1.192.18.4,0,1,15.94,29.39,46.53,43.33,0,0.01
3,1.204.166.3,0,1,12.10,24.36,48.52,41.00,0,0.08
4,1.234.83.26,2,11,57.95,65.15,77.87,70.33,0,29.85
...,...,...,...,...,...,...,...,...,...
5874,ymath.yonsei.ac.kr,0,0,0.17,1.02,2.64,12.94,0,0.02
5875,yotpo-static.com,0,0,1.39,1.69,5.26,13.81,0,0.04
5876,yourpensionmeeting.com,3,12,56.24,67.00,79.39,67.86,0,31.47
5877,zerocap.com,0,0,0.00,0.00,0.00,NaN,<NA>,NaN


In [3]:
tas_path = r"Z:\HTOC\Data_Analytics\Data\Threat Assessment Scores\Threat_Assessment_Scores.xlsx"

# Read the first sheet (sheet index 0) into a DataFrame
tas_df = pd.read_excel(tas_path, sheet_name=0)
tas_df

,Indicator,Last Observed,Indicator Type,Observation Yearly Count,ThreatConnect Rating,Observation Penalty Multiplier,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation
0,101.71.130.99,2026-03-03,Address,139,3,0.992384,0,0,"CMS, DHA, FDA, HHS, HRSA, NIH, OS",Incident:INC9270850,NaN,180,346,462,medium,[2026-03-23] Severity: medium. VT score: 11. C...
1,103.114.96.246,2026-02-26,Address,41,1,0.997753,1,0,"CMS, DHA, NIH, OS, VA",NaN,Mr Hamza Group,170,224,130,low,[2026-03-23] Severity: low. VT score: 1. Conte...
2,103.120.116.162,2026-03-23,Address,55,3,0.996986,0,0,"CMS, DHA, FDA, HRSA, OS, VA",Incident:INC9385644,NaN,740,870,384,medium,[2026-03-23] Severity: medium. VT score: 7. Co...
3,103.136.147.4,2026-02-23,Address,10,5,0.999452,0,0,"DHA, VA",NaN,UNC5537,170,585,320,medium,[2026-03-23] Severity: medium. VT score: 5. Co...
4,103.140.62.43,2026-03-19,Address,14,5,0.999233,0,0,"DHA, VA",NaN,NaN,180,590,145,low,[2026-03-23] Severity: low. VT score: 3. Conte...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2319,198.199.103.243,2026-01-16,Address,1,0,0.999945,1,0,VA,Incident:6755399478001848,NaN,180,368,41,low,Severity: low. VT score: 0. Top drivers: Incid...
2320,w-2@doculivery.com,2026-03-04,EmailAddress,2,4,0.999890,1,0,HHS,Incident:547380,NaN,170,224,29,low,[2026-03-06] Severity: low. VT score not avail...
2321,link.edgepilot.com,2026-03-05,Host,74,1,0.995945,0,0,"CMS, NIH",NaN,NaN,180,229,28,low,[2026-03-06] Severity: low. VT score not avail...
2322,52.87.206.242,2026-01-08,Address,1,1,0.999945,1,0,IHS,NaN,NaN,170,224,27,low,Severity: low. VT score: 0. Top drivers: TC ra...


In [4]:
merged = grouped.merge(tas_df, on="Indicator", how="left")

# Drop rows where all TAS columns are missing
tas_cols = [c for c in merged.columns if c not in grouped.columns]
merged_nonnull = merged.dropna(subset=tas_cols, how="all").copy()

# Fill missing incidents/events and threat actor fields with text

# Columns related to incidents/events
incident_cols = [c for c in merged_nonnull.columns 
                 if "incident" in c.lower() or "event" in c.lower()]

# Columns related to threat actors
threat_actor_cols = [c for c in merged_nonnull.columns 
                     if "threat actor" in c.lower() or "actor" in c.lower()]

merged_nonnull.loc[:, incident_cols] = merged_nonnull[incident_cols].fillna("No known incidents/events")
merged_nonnull.loc[:, threat_actor_cols] = merged_nonnull[threat_actor_cols].fillna("No known threat actors")

merged_nonnull

,Indicator,Frequency (7d),Frequency (30d),Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day,Frequency (1d),Probability: 1-Day,Last Observed,...,Botnet Flag,False Positives,Partners,incidents/events,Threat Actor,CAL Score,ThreatConnect Score,PRISM Score,Severity,Explanation
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,0,0,9.21,18.59,27.13,29.95,0,0.34,2026-01-08,...,1.0,0.0,VA,Incident:30246,No known threat actors,810.0,905.0,177.0,low,Severity: low. Could not pull VT score to calc...
1,1-you.njalla.no,0,2,23.79,39.99,61.14,48.66,0,8.62,2026-02-03,...,0.0,0.0,"NIH, VA",No known incidents/events,Wicked Panda,180.0,416.0,312.0,medium,[2026-03-05] Severity: medium. VT score not av...
4,1.234.83.26,2,11,57.95,65.15,77.87,70.33,0,29.85,2026-02-14,...,0.0,0.0,NaN,No known incidents/events,No known threat actors,180.0,469.0,134.0,low,[2026-03-16] Severity: low. VT score: 3. Top d...
5,1.4.195.14,0,2,21.23,31.84,49.17,45.92,0,5.22,2026-01-17,...,1.0,0.0,"CMS, VA",No known incidents/events,No known threat actors,180.0,229.0,134.0,low,Severity: low. VT score: 1. Top drivers: TOR a...
12,101.168.57.163,0,2,13.25,29.41,63.49,55.77,0,3.87,2026-02-14,...,0.0,0.0,NaN,Incident:INC9407577,No known threat actors,180.0,590.0,389.0,medium,[2026-03-16] Severity: medium. VT score: 11. T...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5832,www.deepseek.com,2,11,63.09,72.95,82.06,74.57,0,24.03,2026-03-20,...,0.0,0.0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",No known incidents/events,No known threat actors,180.0,469.0,50.0,low,[2026-03-23] Severity: low. VT score not avail...
5833,www.deepseek.com.cdn.cloudflare.net,1,4,23.25,32.04,44.13,40.74,0,4.17,2025-11-28,...,0.0,0.0,"CMS, DHA, IHS, NIH, VA",No known incidents/events,No known threat actors,170.0,464.0,293.0,medium,Severity: medium. Could not pull VT score to c...
5862,www.prosinthecity.com,0,1,8.38,16.41,32.61,33.86,0,3.36,2025-11-12,...,0.0,0.0,"CMS, DHA, NIH, OS",Incident:6755399448004157;Incident:546481,No known threat actors,170.0,363.0,417.0,medium,Severity: medium. Could not pull VT score to c...
5865,www.sthda.com,1,8,44.83,58.50,69.67,63.83,0,10.71,2026-03-03,...,0.0,0.0,"CMS, DHA, FDA, HRSA, IHS, NIH, OS, VA",Incident:331131,No known threat actors,170.0,363.0,153.0,low,[2026-03-23] Severity: low. VT score not avail...


In [5]:
from datetime import datetime, timedelta

# Ensure 'Last Observed' or 'Last Observed Indicator' column is present
date_cols = [c for c in merged_nonnull.columns if 'last observed' in c.lower()]
if not date_cols:
    raise ValueError("No column for last observed date found. Check for date column naming.")
last_observed_col = date_cols[0]

# Convert to datetime if not already
merged_nonnull[last_observed_col] = pd.to_datetime(merged_nonnull[last_observed_col], errors="coerce")

cutoff_date = datetime.now() - timedelta(hours=48)
recent_df = merged_nonnull[merged_nonnull[last_observed_col] >= cutoff_date]

if "PRISM Score" in recent_df.columns:
    score_col = "PRISM Score"
else:
    score_cols = [c for c in recent_df.columns if "threat score" in c.lower()]
    if not score_cols:
        raise ValueError("No PRISM Score column found in filtered DataFrame.")
    score_col = score_cols[0]

top_10 = recent_df.sort_values(by=score_col, ascending=False).head(10)
top_10 = top_10[["Indicator", score_col, last_observed_col]]
top_10

,Indicator,PRISM Score,Last Observed
3776,45.148.10.141,842.0,2026-03-23
3948,45.9.148.129,697.0,2026-03-23
3134,213.209.159.158,693.0,2026-03-22
4853,85.93.218.204,686.0,2026-03-22
2429,193.32.162.145,654.0,2026-03-23
101,103.143.10.79,633.0,2026-03-23
767,138.59.233.5,629.0,2026-03-23
2781,2.57.121.112,629.0,2026-03-23
1353,165.154.173.226,628.0,2026-03-23
627,123.58.207.81,625.0,2026-03-23


In [18]:
# Table: top-10 recent indicators with their probabilities

prob_cols = [
    "Indicator",
    score_col,                 # e.g. "PRISM Score"
    "Probability: 1-Day",
    "Probability: 7-Day",
    "Probability: 14-Day",
    "Probability: 30-Day",
    "Probability: 45-Day",
]

# Keep only columns that actually exist
prob_cols = [c for c in prob_cols if c in recent_df.columns]

top_10_probabilities = (
    recent_df[recent_df["Indicator"].isin(top_10["Indicator"])]
    [prob_cols]
    .drop_duplicates(subset=["Indicator"])
    .sort_values(by=score_col, ascending=False)
    .reset_index(drop=True)
)
# Drop the PRISM Score column if it exists, as instructed
if "PRISM Score" in top_10_probabilities.columns:
    top_10_probabilities = top_10_probabilities.drop(columns=["PRISM Score"])

# Convert probability columns to percentages (if not already)
probability_cols = [col for col in top_10_probabilities.columns if col.startswith("Probability")]
for col in probability_cols:
    # If values are consistently <= 1, assume they are fractions and convert
    if top_10_probabilities[col].dropna().max() <= 1.0:
        top_10_probabilities[col] = top_10_probabilities[col] * 100
# Instead of converting probabilities to percentages, ensure they are decimals (fractions between 0 and 1)
probability_cols = [col for col in top_10_probabilities.columns if col.startswith("Probability")]
for col in probability_cols:
    # If values are mostly >= 1.0, assume they are percentages and convert to decimals
    if top_10_probabilities[col].dropna().min() >= 1.0:
        top_10_probabilities[col] = top_10_probabilities[col] / 100

top_10_probabilities

,Indicator,Probability: 1-Day,Probability: 7-Day,Probability: 14-Day,Probability: 30-Day,Probability: 45-Day
0,45.148.10.141,0.4322,0.7281,0.7733,0.8323,0.7347
1,45.9.148.129,0.3618,0.7756,0.9073,0.9826,0.8176
2,213.209.159.158,0.4189,0.8897,0.9699,0.9966,0.7671
3,85.93.218.204,0.0755,0.1860,0.3526,0.5932,0.5485
4,193.32.162.145,0.5683,0.9138,0.9641,0.9862,0.8889
5,103.143.10.79,0.5291,0.8547,0.9614,0.9964,0.7596
6,2.57.121.112,0.5769,0.9847,0.9974,0.9997,0.8966
7,138.59.233.5,0.5117,0.8517,0.9146,0.9705,0.8218
8,165.154.173.226,0.6100,0.8903,0.9177,0.9384,0.8806
9,123.58.207.81,0.6723,0.9724,0.9869,0.9945,0.9270


Threat Recurrence Risk Idex

In [7]:
# Total TRRI scores across all (assumed active) indicators
# Expected Value = Impact × Likelihood
trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

total_trri = {col: merged_nonnull[col].sum(skipna=True) for col in trri_cols}

total_trri

{}

In [8]:
import pandas as pd

preferred_score_col = "PRISM Score"

if preferred_score_col in merged_nonnull.columns:
    score_col = preferred_score_col
else:
    score_cols = [c for c in merged_nonnull.columns if "threat score" in c.lower()]
    if not score_cols:
        raise ValueError("No PRISM Score column found; check TAS column names.")
    score_col = score_cols[0]

merged_nonnull[score_col] = pd.to_numeric(merged_nonnull[score_col], errors="coerce")

prob_map = {
    "1d": "Probability: 1-Day",
    "7d": "Probability: 7-Day",
    "14d": "Probability: 14-Day",
    "30d": "Probability: 30-Day",
    "45d": "Probability: 45-Day",
}

for suffix, pcol in prob_map.items():
    if pcol not in merged_nonnull.columns:
        continue

    merged_nonnull[pcol] = pd.to_numeric(merged_nonnull[pcol], errors="coerce")

    max_prob = merged_nonnull[pcol].max(skipna=True)

    # If max > 1, assume percentages like 12, 25, 68
    if pd.notna(max_prob) and max_prob > 1:
        prob_series = merged_nonnull[pcol] / 100.0
    else:
        prob_series = merged_nonnull[pcol]

    merged_nonnull[f"TRRI_{suffix}"] = (
        merged_nonnull[score_col] * prob_series
    ).round(1)

display_cols = ["Indicator", score_col] + [f"TRRI_{k}" for k in prob_map if f"TRRI_{k}" in merged_nonnull.columns]
merged_nonnull[display_cols]

,Indicator,PRISM Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,177.0,0.6,16.3,32.9,48.0,53.0
1,1-you.njalla.no,312.0,26.9,74.2,124.8,190.8,151.8
4,1.234.83.26,134.0,40.0,77.7,87.3,104.3,94.2
5,1.4.195.14,134.0,7.0,28.4,42.7,65.9,61.5
12,101.168.57.163,389.0,15.1,51.5,114.4,247.0,216.9
...,...,...,...,...,...,...,...
5832,www.deepseek.com,50.0,12.0,31.5,36.5,41.0,37.3
5833,www.deepseek.com.cdn.cloudflare.net,293.0,12.2,68.1,93.9,129.3,119.4
5862,www.prosinthecity.com,417.0,14.0,34.9,68.4,136.0,141.2
5865,www.sthda.com,153.0,16.4,68.6,89.5,106.6,97.7


In [9]:
# Step 5 — Create TRRI risk zones per horizon using percentiles

import pandas as pd

# Percentile cut-points:
# 0–40%   -> Low
# 40–70%  -> Moderate
# 70–90%  -> High
# 90–100% -> Critical
quantiles = [0.0, 0.4, 0.7, 0.9, 1.0]
labels = [
    "Low Recurrence Risk",        # bottom 40%
    "Moderate Recurrence Risk",   # next 30%
    "High Recurrence Risk",       # next 20%
    "Critical Recurrence Risk",   # top 10%
]

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

for col in trri_cols:
    risk_col = col.replace("TRRI_", "Risk_")
    # Use qcut on the TRRI values to assign percentile-based risk
    merged_nonnull[risk_col] = pd.qcut(
        merged_nonnull[col],
        q=quantiles,
        labels=labels,
        duplicates="drop"  # handle cases where some bins collapse
    )

# Quick look at indicator, PRISM Score, TRRI, and risk zones
cols_to_show = (
    ["Indicator", "PRISM Score"]
    + trri_cols
    + [c for c in merged_nonnull.columns if c.startswith("Risk_")]
)
merged_nonnull[cols_to_show]

,Indicator,PRISM Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d,Risk_1d,Risk_7d,Risk_14d,Risk_30d,Risk_45d
0,0F2C5C39494F15B7EE637AD5B6B5D00A3E2F407B4F27D1...,177.0,0.6,16.3,32.9,48.0,53.0,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
1,1-you.njalla.no,312.0,26.9,74.2,124.8,190.8,151.8,Moderate Recurrence Risk,Low Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk
4,1.234.83.26,134.0,40.0,77.7,87.3,104.3,94.2,Moderate Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5,1.4.195.14,134.0,7.0,28.4,42.7,65.9,61.5,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
12,101.168.57.163,389.0,15.1,51.5,114.4,247.0,216.9,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk
...,...,...,...,...,...,...,...,...,...,...,...,...
5832,www.deepseek.com,50.0,12.0,31.5,36.5,41.0,37.3,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5833,www.deepseek.com.cdn.cloudflare.net,293.0,12.2,68.1,93.9,129.3,119.4,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5862,www.prosinthecity.com,417.0,14.0,34.9,68.4,136.0,141.2,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk
5865,www.sthda.com,153.0,16.4,68.6,89.5,106.6,97.7,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk,Low Recurrence Risk


In [10]:
# Indicators from your 48-hour / top-threat subset
top_inds = top_10["Indicator"].unique()

# Filter the full merged DF to just those indicators
recent_top_trri = merged_nonnull[merged_nonnull["Indicator"].isin(top_inds)].copy()

# Build a TRRI-focused view, sorted by PRISM Score
cols = [
    "Indicator",
    "PRISM Score",
    "TRRI_1d",
    "TRRI_7d",
    "TRRI_14d",
    "TRRI_30d",
    "TRRI_45d",
    "Risk_1d",
    "Risk_7d",
    "Risk_14d",
    "Risk_30d",
    "Risk_45d",
]
existing_cols = [c for c in cols if c in recent_top_trri.columns]

recent_top_trri = (
    recent_top_trri[existing_cols]
    .sort_values("PRISM Score", ascending=False)
    .reset_index(drop=True)
)

recent_top_trri

,Indicator,PRISM Score,TRRI_1d,TRRI_7d,TRRI_14d,TRRI_30d,TRRI_45d,Risk_1d,Risk_7d,Risk_14d,Risk_30d,Risk_45d
0,45.148.10.141,842.0,363.9,613.1,651.1,700.8,618.6,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
1,45.9.148.129,697.0,252.2,540.6,632.4,684.9,569.9,High Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
2,213.209.159.158,693.0,290.3,616.6,672.1,690.6,531.6,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
3,85.93.218.204,686.0,51.8,127.6,241.9,406.9,376.3,Moderate Recurrence Risk,Moderate Recurrence Risk,Moderate Recurrence Risk,High Recurrence Risk,High Recurrence Risk
4,193.32.162.145,654.0,371.7,597.6,630.5,645.0,581.3,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
5,103.143.10.79,633.0,334.9,541.0,608.6,630.7,480.8,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
6,2.57.121.112,629.0,362.9,619.4,627.4,628.8,564.0,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
7,138.59.233.5,629.0,321.9,535.7,575.3,610.4,516.9,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
8,165.154.173.226,628.0,383.1,559.1,576.3,589.3,553.0,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk
9,123.58.207.81,625.0,420.2,607.8,616.8,621.6,579.4,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk,Critical Recurrence Risk


In [11]:
# Table: top-10 recent indicators with their explanations
explanation_cols = ["Indicator", score_col, "Explanation"]
explanation_cols = [c for c in explanation_cols if c in recent_df.columns]

top_10_explanations = (
    recent_df[recent_df["Indicator"].isin(top_10["Indicator"])]
    [explanation_cols]
    .drop_duplicates(subset=["Indicator"])
    .sort_values(by=score_col, ascending=False)
    .reset_index(drop=True)
)

top_10_explanations

,Indicator,PRISM Score,Explanation
0,45.148.10.141,842.0,[2026-03-23] Severity: critical. VT score: 16....
1,45.9.148.129,697.0,[2026-03-23] Severity: high. VT score: 16. Con...
2,213.209.159.158,693.0,[2026-03-23] Severity: high. VT score: 18. Con...
3,85.93.218.204,686.0,[2026-03-23] Severity: high. VT score: 12. Con...
4,193.32.162.145,654.0,[2026-03-23] Severity: high. VT score: 16. Con...
5,103.143.10.79,633.0,[2026-03-23] Severity: high. VT score: 15. Con...
6,2.57.121.112,629.0,[2026-03-23] Severity: high. VT score: 16. Con...
7,138.59.233.5,629.0,[2026-03-23] Severity: high. VT score: 15. Con...
8,165.154.173.226,628.0,[2026-03-23] Severity: high. VT score: 15. Con...
9,123.58.207.81,625.0,[2026-03-23] Severity: high. VT score: 15. Con...


import os

trri_dir = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI"
os.makedirs(trri_dir, exist_ok=True)

out_path = os.path.join(trri_dir, "TRRI_merged_nonnull_2025-06-16.xlsx")  # adjust date/name as needed
merged_nonnull.to_excel(out_path, index=False)

out_path

In [12]:

# DataFrames of the top 10 indicators for each TRRI column, along with their PRISM Score
top_10_trri_dfs = {}

for col in trri_cols:
    # Get top 10 by TRRI value, include Indicator and PRISM Score
    df_top10 = (
        merged_nonnull[["Indicator", "PRISM Score", col]]
        .sort_values(col, ascending=False)
        .head(10)
        .reset_index(drop=True)
    )
    top_10_trri_dfs[col] = df_top10
    print(f"\n=== Top 10 for {col} ===")
    display(df_top10)





=== Top 10 for TRRI_1d ===


,Indicator,PRISM Score,TRRI_1d
0,207.90.244.3,584.0,445.5
1,64.225.74.178,597.0,443.6
2,198.235.24.186,585.0,441.9
3,205.210.31.163,600.0,440.7
4,123.58.207.81,625.0,420.2
5,80.94.92.171,583.0,410.7
6,139.19.117.131,610.0,409.6
7,207.90.244.13,620.0,398.1
8,183.224.237.233,580.0,396.1
9,198.235.24.193,557.0,395.2



=== Top 10 for TRRI_7d ===


,Indicator,PRISM Score,TRRI_7d
0,2.57.121.112,629.0,619.4
1,213.209.159.158,693.0,616.6
2,45.148.10.141,842.0,613.1
3,123.58.207.81,625.0,607.8
4,2.57.121.25,609.0,603.8
5,193.32.162.145,654.0,597.6
6,139.19.117.131,610.0,596.7
7,176.65.134.22,727.0,591.3
8,205.210.31.163,600.0,589.9
9,64.225.74.178,597.0,587.9



=== Top 10 for TRRI_14d ===


,Indicator,PRISM Score,TRRI_14d
0,213.209.159.158,693.0,672.1
1,176.65.134.22,727.0,652.6
2,45.148.10.141,842.0,651.1
3,45.9.148.129,697.0,632.4
4,193.32.162.145,654.0,630.5
5,2.57.121.112,629.0,627.4
6,123.58.207.81,625.0,616.8
7,103.143.10.79,633.0,608.6
8,2.57.121.25,609.0,606.8
9,139.19.117.131,610.0,606.2



=== Top 10 for TRRI_30d ===


,Indicator,PRISM Score,TRRI_30d
0,45.148.10.141,842.0,700.8
1,213.209.159.158,693.0,690.6
2,176.65.134.22,727.0,687.2
3,45.9.148.129,697.0,684.9
4,193.32.162.145,654.0,645.0
5,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,980.0,638.8
6,103.143.10.79,633.0,630.7
7,2.57.121.112,629.0,628.8
8,123.58.207.81,625.0,621.6
9,165.154.10.165,623.0,618.8



=== Top 10 for TRRI_45d ===


,Indicator,PRISM Score,TRRI_45d
0,45.148.10.141,842.0,618.6
1,AE8293EA164A55787E48E4373C875EC8C2AD2AF47C22A9...,980.0,598.4
2,193.32.162.145,654.0,581.3
3,123.58.207.81,625.0,579.4
4,176.65.134.22,727.0,578.8
5,45.9.148.129,697.0,569.9
6,2.57.121.112,629.0,564.0
7,139.19.117.131,610.0,560.2
8,165.154.10.165,623.0,553.2
9,165.154.173.226,628.0,553.0


In [13]:
# Total TRRI scores across all (assumed active) indicators

trri_cols = [c for c in merged_nonnull.columns if c.startswith("TRRI_")]

total_trri = {col: merged_nonnull[col].sum(skipna=True) for col in trri_cols}

# Turn into a clean DataFrame
total_trri_df = (
    pd.DataFrame(list(total_trri.items()), columns=["TRRI_Horizon", "Total_TRRI"])
    .sort_values("TRRI_Horizon")
    .reset_index(drop=True)
)

total_trri_df

,TRRI_Horizon,Total_TRRI
0,TRRI_14d,418361.1
1,TRRI_1d,202994.1
2,TRRI_30d,488014.2
3,TRRI_45d,433028.3
4,TRRI_7d,357096.9


# Save current TRRI totals to CSV
out_path = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI\TRRI_totals_2025-06-16.csv"  # adjust name/date as needed
total_trri_df.to_csv(out_path, index=False)
out_path

import os
import glob
import pandas as pd

trri_dir = r"C:\Users\jaskew\Documents\HTOC\notebooks\ThreaRecurrenceRiskIndex\TRRI"

# Get all TRRI total CSVs in the folder
csv_files = glob.glob(os.path.join(trri_dir, "TRRI_totals_*.csv"))
if len(csv_files) < 2:
    raise ValueError("Need at least two TRRI_totals_*.csv files to compare.")

# Sort by filename (assumes date is in the name and sortable, e.g. TRRI_totals_2025-06-16.csv)
csv_files_sorted = sorted(csv_files)

prev_path = csv_files_sorted[-2]  # previous file
curr_path = csv_files_sorted[-1]  # current file

prev_df = pd.read_csv(prev_path)
curr_df = pd.read_csv(curr_path)

compare = prev_df.merge(
    curr_df,
    on="TRRI_Horizon",
    suffixes=("_prev", "_curr")
)

compare["Pct_Change"] = (
    (compare["Total_TRRI_curr"] - compare["Total_TRRI_prev"])
    / compare["Total_TRRI_prev"]
) * 100

compare